In [76]:

import ojas_feb0126_find_best_run as fbr
import json
import numpy as np

In [77]:
root_path = "/mnt/home/donti-group-shared/ojas/pfdelta/runs/gen_feb0126"

cases = ["case14", "case30", "case57", "case118", "case500"]
combined_cases = ["case14_30_118", "case30_57_118", "case30_57_500", "case57_118_500"]
tasks = ["task31", "task32", "task33", "task34"]
models = ["graphconv", "powerflownet"]
error_key = "PBL Mean"

In [78]:
results = {}

def add_result(best_runs, task, model, case):
    if results.get(task) is None:
        results[task] = {}
    if results[task].get(model) is None:
        results[task][model] = {}
        
    results[task][model][case] = best_runs

In [ ]:
for model in models:
    # print("---------------TASK 1")
    for task in [tasks[0]]:
        for case in cases:
            folder = f"{root_path}/{task}/{model}/{case}"
            best_runs = fbr.find_best_run(folder, error_key)
            add_result(best_runs, task, model, case)

    # print("---------------TASK 2 AND 3")
    for task in [tasks[1], tasks[2]]:
        folder = f"{root_path}/{task}/{model}/"
        best_runs = fbr.find_best_run(folder, error_key)
        add_result(best_runs, task, model, case)

    for task in [tasks[3]]:
        for case in combined_cases:
            folder = f"{root_path}/{task}/{model}/{case}"
            best_runs = fbr.find_best_run(folder, error_key)
            add_result(best_runs, task, model, case)

---------------TASK 1
---------------TASK 2 AND 3
---------------TASK 1
---------------TASK 2 AND 3


In [81]:
ROWS = ["14", "30", "57", "118", "500", "14_30_57", "118_500", "14_30_118", "30_57_118", "30_57_500", "57_118_500"]
COLS = ROWS.copy()[:5]

row_to_cols = {}

for row in ROWS:
    row_to_cols[row] = [COLS]
    
row_idx = {r: i for i, r in enumerate(ROWS)}
col_idx = {c: j for j, c in enumerate(COLS)}

In [82]:
def get_row_col(train_case, test_case):
    if test_case.startswith("case"):
        test_case = test_case[4:]
    if train_case.startswith("case"):
        train_case = train_case[4:]
    return [row_idx[train_case], col_idx[test_case]]

In [83]:
graphconv_means = np.full((len(ROWS), len(COLS)), np.nan, dtype=float)
graphconv_stds = np.full((len(ROWS), len(COLS)), np.nan, dtype=float)
powerflownet_means = np.full((len(ROWS), len(COLS)), np.nan, dtype=float)
powerflownet_stds = np.full((len(ROWS), len(COLS)), np.nan, dtype=float)

In [84]:
for task, task_values in results.items():
    for model, model_values in task_values.items():
        # print(model_values.items())
        for train_case, train_values in model_values.items():
            # print(train_values)

            print("Processing:", task, model, train_case)

            summaries = train_values[4]

            # print(summaries)

            if summaries is None:
                continue

            mean_err_per_test_case_per_seed = {
                "case14": [],
                "case30": [],
                "case57": [],
                "case118": [],
                "case500": [],
            }

            # go through all of the seeds
            # for each seed, get the summary
            for summary in summaries:
                if summary is None:
                    [mean_err_per_test_case_per_seed[i].append(np.nan) for i in cases]
                    continue

                val = summary["val"]

                # go through all of the validation sets
                # skip the first one
                # take the pbl mean, put it into the appropriate spot in the list
                # depending on the test_case
                # we shoud have one entry per test case per seed
                for i in range(1, len(val)):
                    test_case = cases[i - 1]
                    val_dict = val[i]
                    err = val_dict[error_key]

                    mean_err_per_test_case_per_seed[test_case].append(err)

            print(mean_err_per_test_case_per_seed)

            for test_case in mean_err_per_test_case_per_seed.keys():
                # calculate mean and std for the err for each test_case
                errs = mean_err_per_test_case_per_seed[test_case]
                mean_err = np.mean(errs)
                std_err = np.std(errs)

                # these should be the same
                # best_err_mean = train_values[2]
                # best_err_std = train_values[3]
                # print("this case best err mean std: ", best_err_mean, best_err_std)
                # print("computed case mean std: ", mean_err, std_err)
                
                r, c = get_row_col(train_case, test_case)
                # print(r, c)
                # print(mean_err)

                if model == "graphconv":
                    graphconv_means[r, c] = mean_err
                    graphconv_stds[r, c] = std_err
                elif model == "powerflownet":
                    powerflownet_means[r, c] = mean_err
                    powerflownet_stds[r, c] = std_err

Processing: task31 graphconv case14
{'case14': [0.30822585154982174, nan, nan], 'case30': [1.244473768683041, nan, nan], 'case57': [12.291698354833267, nan, nan], 'case118': [42.319136092242076, nan, nan], 'case500': [196.82909779268152, nan, nan]}
Processing: task31 graphconv case30
{'case14': [3.341984089683084, nan, nan], 'case30': [0.2642299052547006, nan, nan], 'case57': [24.915404196346508, nan, nan], 'case118': [82.61881929285386, nan, nan], 'case500': [646.7537138097426, nan, nan]}
Processing: task31 graphconv case57
{'case14': [nan, nan, nan], 'case30': [nan, nan, nan], 'case57': [nan, nan, nan], 'case118': [nan, nan, nan], 'case500': [nan, nan, nan]}
Processing: task31 graphconv case118
{'case14': [nan, nan, nan], 'case30': [nan, nan, nan], 'case57': [nan, nan, nan], 'case118': [nan, nan, nan], 'case500': [nan, nan, nan]}
Processing: task31 graphconv case500
{'case14': [nan, nan, nan], 'case30': [nan, nan, nan], 'case57': [nan, nan, nan], 'case118': [nan, nan, nan], 'case500'

In [85]:
graphconv_means

array([[nan, nan, nan, nan, nan],
       [nan, nan, nan, nan, nan],
       [nan, nan, nan, nan, nan],
       [nan, nan, nan, nan, nan],
       [nan, nan, nan, nan, nan],
       [nan, nan, nan, nan, nan],
       [nan, nan, nan, nan, nan],
       [nan, nan, nan, nan, nan],
       [nan, nan, nan, nan, nan],
       [nan, nan, nan, nan, nan],
       [nan, nan, nan, nan, nan]])